In [69]:
import mlflow
import pandas as pd
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os

In [71]:
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv')
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [72]:
df.drop(columns=['tweet_id'])

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...
...,...,...
39995,neutral,@JohnLloydTaylor
39996,love,Happy Mothers Day All my love
39997,love,Happy Mother's Day to all the mommies out ther...
39998,happiness,@niariley WASSUP BEAUTIFUL!!! FOLLOW ME!! PEE...


In [73]:
# data preprocessing

# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise

<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
<>:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
/var/folders/5b/6p1qx5k11xz64d7d08l0vmg80000gn/T/ipykernel_2515/93639238.py:32: SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\s"? A raw string is also an option.
  text = re.sub('\s+', ' ', text).strip()


In [74]:
# 1. Filter the dataset to only happiness and sadness
df = df[df['sentiment'].isin(['happiness', 'sadness'])].reset_index(drop=True)

# 2. Run your text normalization on the whole dataframe
df = normalize_text(df)

# 3. NOW map the sentiments to 0 and 1
df['sentiment'] = df['sentiment'].replace({'sadness': 0, 'happiness': 1})

# 4. Force the sentiment column to be integers just to be 100% safe
df['sentiment'] = df['sentiment'].astype(int)

# Check the final output before training your model
print(df['sentiment'].value_counts())
df.head()

sentiment
1    5209
0    5165
Name: count, dtype: int64


,tweet_id,sentiment,content
0,1956967666,0,layin n bed headache ughhhh waitin call
1,1956967696,0,funeral ceremony gloomy friday
2,1956968487,0,sleep im not thinking old friend want married ...
3,1956969035,0,charviray charlene love miss
4,1956969172,0,kelcouch sorry least friday


In [ ]:
import dagshub
dagshub.init(repo_owner='arshpreetsingh-01', repo_name='mlops-mini-project', mlflow=True)

mlflow.set_experiment("Logistic Regressio Baseline")
mlflow.set_experiment("Logistic Regressio Baseline")

Initialized MLflow to track repo "arshpreetsingh-01/mlops-mini-project"

Repository arshpreetsingh-01/mlops-mini-project initialized!

<Experiment: artifact_location='mlflow-artifacts:/0a725b95993f42869f580567781c6412', creation_time=1789795136184, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1789795136184, lifecycle_stage='active', name='Logistic Regressio Baseline', tags={}, trace_location=None, workspace='default'>

In [76]:
vectorizer = CountVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['content'])
y = df['sentiment']

In [77]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [81]:
with mlflow.start_run():
    # Log preprocessing parameters
    mlflow.log_param("vectorizer", "Bag of Words")
    mlflow.log_param("num_features", 1000)
    mlflow.log_param("test_size", 0.2)
    
    # Model building and training
    model = LogisticRegression()
    model.fit(X_train, y_train)
    
    # Log model parameters
    mlflow.log_param("model", "Logistic Regression")
    
    # Model evaluation
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    # Log evaluation metrics
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)
    
    # Log model (This creates a FOLDER named 'model', not a single file)
    mlflow.sklearn.log_model(model, "model")

    # Save and log the notebook directly (Removed --execute)
    import os
    notebook_path = "exp1_baseline_model.ipynb"
    
    if os.path.exists(notebook_path):
        mlflow.log_artifact(notebook_path)
    else:
        print(f"Warning: Could not find {notebook_path} to log as artifact.")
    
    # Print the results for verification
    print(f"Accuracy: {accuracy}")
    print(f"Precision: {precision}")
    print(f"Recall: {recall}")
    print(f"F1 Score: {f1}")

2026/09/19 14:14:56 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Accuracy: 0.7773493975903615
Precision: 0.7692307692307693
Recall: 0.7783251231527094
F1 Score: 0.7737512242899118
🏃 View run dashing-gnu-971 at: https://dagshub.com/arshpreetsingh-01/mlops-mini-project.mlflow/#/experiments/0/runs/004209414f4c4c34966b63ab25c2992a
🧪 View experiment at: https://dagshub.com/arshpreetsingh-01/mlops-mini-project.mlflow/#/experiments/0
